## Домашнее задание №2
### MADE-DS-22, Сергей Нефедов

In [1]:
import pickle
import re

import pandas as pd
import numpy as np

1. Прочитайте и проанализируйте данные, выберите турниры, в которых есть данные о
составах команд и повопросных результатах (поле mask в results.pkl). Для унификации
предлагаю:
* взять в тренировочный набор турниры с dateStart из 2019 года;
* в тестовый — турниры с dateStart из 2020 года.

In [2]:
with open('results.pkl', 'rb') as f:
    results = pickle.load(f)

In [3]:
with open('players.pkl', 'rb') as f:
    players = pickle.load(f)

In [4]:
with open('tournaments.pkl', 'rb') as f:
    tournaments = pickle.load(f)

In [5]:
[(x, y) for x, y in players.items()][::100000]

[(1, {'id': 1, 'name': 'Алексей', 'patronymic': None, 'surname': 'Абабилов'}),
 (112708,
  {'id': 112708,
   'name': 'Назар',
   'patronymic': 'Батырович',
   'surname': 'Бекмурадов'}),
 (220277,
  {'id': 220277,
   'name': 'Юлия',
   'patronymic': 'Геннадьевна',
   'surname': 'Лежнина'})]

In [6]:
players_names = {player['id']: player['name'] + ' ' + player['surname'] for player in players.values()}

In [7]:
len(players_names)

204063

In [8]:
[(x, y) for x, y in tournaments.items()][::100000]

[(1,
  {'id': 1,
   'name': 'Чемпионат Южного Кавказа',
   'dateStart': '2003-07-25T00:00:00+04:00',
   'dateEnd': '2003-07-27T00:00:00+04:00',
   'type': {'id': 2, 'name': 'Обычный'},
   'season': '/seasons/1',
   'orgcommittee': [],
   'synchData': None,
   'questionQty': None})]

In [9]:
tournament_names = {tournament['id']: tournament['name'] for tournament in tournaments.values()}

In [10]:
tournaments_train = set([tournament['id'] for tournament in tournaments.values() if tournament['dateStart'][:4] == '2019'])
tournaments_test = set([tournament['id'] for tournament in tournaments.values() if tournament['dateStart'][:4] == '2020'])

In [11]:
len(tournaments_train)

687

In [12]:
len(tournaments_test)

418

In [13]:
teams_dict = {}
results_train = {}
results_test = {}

for tournament_id, teams in results.items():
    tournament_results = {}
    for team in teams:
        team_mask = team.get('mask')
        team_members = [player['player']['id'] for player in team['teamMembers']]
        
        if team_mask is None or re.findall('[^01X?]', team_mask) or not team_members:
            continue
            
        team_mask = team_mask.replace('X', '').replace('?', '')
          
        team_id = team['team']['id']
        team_name = team['team']['name']
        teams_dict[team_id] = team_name
        
        tournament_results[team_id] = {}
        tournament_results[team_id]['mask'] = team_mask
        tournament_results[team_id]['players'] = team_members
        
    if len(set(list(map(len, [team['mask'] for team in tournament_results.values()])))) == 1:  
        if tournament_id in tournaments_train:
            results_train[tournament_id] = tournament_results
        elif tournament_id in tournaments_test:
            results_test[tournament_id] = tournament_results

2. Постройте baseline-модель на основе линейной или логистической регрессии, которая будет обучать рейтинг-лист игроков.

In [14]:
train = []

max_question_id = 0

for tournament_id, teams in results_train.items():
    for team_id, team in teams.items():
        mask = np.array([np.int32(answer) for answer in team['mask']])
        players = team['players']
        questions = np.tile(np.arange(max_question_id, max_question_id + len(mask)), len(players))
        answers = np.array(np.meshgrid(players, mask)).T.reshape(-1, 2)
        answers = np.hstack([
            np.repeat(tournament_id, len(questions)).reshape(-1, 1),
            np.repeat(team_id, len(questions)).reshape(-1, 1),
            answers, 
            questions.reshape(-1, 1)]
        )
        train.append(answers)
        
    max_question_id += len(mask)
        
train = np.vstack(train).astype(np.int32)
train = pd.DataFrame(train, 
                     columns = ['tournament_id', 'team_id', 'player_id', 'answer', 'question_id'])

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder


ohe = OneHotEncoder(handle_unknown='ignore')

X_train = ohe.fit_transform(train[['player_id', 'question_id']])
y_train = train['answer']

In [16]:
model = LogisticRegression(random_state=42, n_jobs=-1)

model.fit(X_train, y_train)

LogisticRegression(n_jobs=-1, random_state=42)

In [17]:
unique_players = np.unique(train['player_id'])
unique_questions = np.unique(train['question_id'])
                    
rating = pd.DataFrame({'player_id': unique_players,
                       'strength': model.coef_[0][:len(unique_players)]})
rating['name'] = rating['player_id'].map(players_names)

In [18]:
rating.sort_values(by='strength', ascending=False).reset_index(drop=True).head(20)

,player_id,strength,name
0,27403,3.881589,Максим Руссо
1,4270,3.648743,Александра Брутер
2,28751,3.420192,Иван Семушин
3,27822,3.356092,Михаил Савченков
4,20691,3.310318,Станислав Мереминский
5,30152,3.285179,Артём Сорожкин
6,30270,3.270940,Сергей Спешков
7,18036,3.235923,Михаил Левандовский
8,22935,3.175416,Илья Новиков
9,56647,3.155013,Наталья Горелова


3. Качество рейтинг-системы оценивается качеством предсказаний результатов турниров. Но сами повопросные результаты наши модели предсказывать вряд ли смогут, ведь неизвестно, насколько сложными окажутся вопросы в будущих турнирах; да и не нужны эти предсказания сами по себе. Поэтому:
* предложите способ предсказать результаты нового турнира с известными составами, но неизвестными вопросами, в виде ранжирования команд;
* в качестве метрики качества на тестовом наборе давайте считать ранговые корреляции Спирмена и Кендалла (их можно взять в пакете scipy ) между реальным ранжированием в результатах турнира и предсказанным моделью, усреднённые по тестовому множеству турниров

In [19]:
from scipy.stats import spearmanr  # корреляция Спирмена
from scipy.stats import kendalltau  # корреляция Кендалла

Оценим силу команд как вероятность того, что хотя бы один участник ответит верно на 1 вопрос:

In [20]:
players_train = set(unique_players)
questions_train = set(unique_questions)

test = []
for tournament_id, teams in results_test.items():
    for team_id, team in teams.items():
        mask = np.array([np.int32(answer) for answer in team['mask']])
        for player_id in team['players']:
            if player_id not in players_train: 
                continue
            
            test.append((tournament_id, team_id, player_id, -1, sum(mask), len(mask))) 
        
test = np.vstack(test).astype(np.int32)
test = pd.DataFrame(test, 
                    columns = ['tournament_id', 'team_id', 'player_id', 'question_id', 'n_true', 'n_total'])

In [21]:
X_test = test[['player_id', 'question_id']]
X_test = ohe.transform(X_test)

preds = model.predict_proba(X_test)[:, 1]

In [22]:
def compute_scores(data, preds):
    data['pred'] = preds
    data['score'] = data.groupby(['tournament_id', 'team_id'])['pred'].transform(lambda x: 1 - np.prod(1 - x))
    rating = data[['tournament_id', 'team_id', 'n_true', 'score']].drop_duplicates().reset_index(drop=True)
    
    # Считаем реальный рейтинг команд
    rating = rating.sort_values(by=['tournament_id', 'n_true'], ascending=False)
    rating['real_rank'] = rating.groupby('tournament_id')['n_true'].transform(lambda x: np.arange(1, len(x) + 1))
    
    # Считаем предсказанный рейтинг
    rating = rating.sort_values(by=['tournament_id', 'score'], ascending=False)
    rating['pred_rank'] = rating.groupby('tournament_id')['score'].transform(lambda x: np.arange(1, len(x) + 1))

    rating = rating.astype(np.int32)
    
    print(f"Корреляция Спирмена: {rating.groupby('tournament_id').apply(lambda x: spearmanr(x['real_rank'], x['pred_rank']).correlation).mean()}")
    print(f"Корреляция Кендалла: {rating.groupby('tournament_id').apply(lambda x: kendalltau(x['real_rank'], x['pred_rank']).correlation).mean()}")
    

compute_scores(test, preds)

Корреляция Спирмена: 0.7749321369618384
Корреляция Кендалла: 0.6053228327790882


4. Теперь главное: ЧГК — это всё-таки командная игра. Поэтому:
* предложите способ учитывать то, что на вопрос отвечают сразу несколько игроков; скорее всего, понадобятся скрытые переменные; не стесняйтесь делать упрощающие предположения, но теперь переменные “игрок X ответил на вопрос Y” при условии данных должны стать зависимыми для игроков одной и той же команды;
* разработайте EM-схему для обучения этой модели, реализуйте её в коде;
* обучите несколько итераций, убедитесь, что целевые метрики со временем растут (скорее всего, ненамного, но расти должны), выберите лучшую модель, используя целевые метрики.

Пусть __X__ - событие того, что игрок ответил на вопрос, __Y__ - команда ответила на вопрос.

Тогда $ P(X|Y) = \frac{P(Y|X) * P(X)}{P(Y)} $

P(Y|X) = 1 - если игрок правильно отвели на вопрос, то и следовательно команда ответила правильно. Поэтому $ P(X|Y) = \frac{P(X)}{P(Y)} $

На __E-шаге__ оцениваем $ P(X|Y) $, а на __M-шаге__ обучаем логистическую регрессию на этом таргете.

In [23]:
from tqdm import tqdm
from scipy import sparse as sp
from scipy.special import expit


def log_loss(y_true, y_pred):
    return - np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


class EMClassifier:
    
    def __init__(self, w=None, lr=25, n_iter=10, batch_size=5000, verbose=1):
        self.w = w
        self.lr = lr
        self.n_iter = n_iter
        self.batch_size = batch_size
        self.verbose = 1
        
    def _add_intercept(self, X):
        return sp.hstack((np.ones((X.shape[0], 1)), X), format='csr')
    
    def _init_w(self, dim):
        self.w = np.random.randn(dim)
        
    def _E_step(self, data, preds):
        team_strength = pd.DataFrame({'team_id': data['team_id'],
                                      'question_id': data['question_id'], 
                                      'team_strength': 1 - preds})
        team_strength = team_strength.groupby(['team_id', 'question_id']).agg({'team_strength': 'prod'}).reset_index()
        team_strength['team_strength'] = 1 - team_strength['team_strength']
        team_strength = data[['team_id', 'question_id']].merge(team_strength)
        y = np.clip(preds / team_strength['team_strength'], 0, 1).values
        y[data['answer'] == 0] = 0
        return y
        
    def _M_step(self, X, y):
        min_loss = np.inf
        indices = np.arange(X.shape[0])
        for _ in range(100):
            indices = np.random.permutation(indices)
            for batch_idx in np.array_split(indices, len(indices) // self.batch_size):
                x_batch, y_batch = X[batch_idx], y[batch_idx]
                grad = x_batch.T.dot(self.predict(x_batch) - y_batch) / len(y_batch)
                self.w -= self.lr * grad
                
            cur_loss = log_loss(y, self.predict(X))
            if min_loss - cur_loss < 1e-6:
                break
                
            min_loss = cur_loss
                
    def fit(self, X_train, train_data, X_test=None, test_data=None):
        X_train = self._add_intercept(X_train)
        if self.w is None or len(self.w) != X_train.shape[1]:
            self._init_w(X_train.shape[1])
        
        for iter_ in tqdm(range(self.n_iter)): 
            preds = self.predict(X_train)
            y = self._E_step(train_data, preds)
            self._M_step(X_train, y)
            if self.verbose is not None and X_test is not None and test_data is not None and iter_ % self.verbose == 0:
                compute_scores(test_data, self.predict(X_test))
                         
    def predict(self, X):
        if self.w is None:
            raise ValueError('Model is not fitted yet!')
        if len(self.w) != X.shape[1]:
            X = self._add_intercept(X)
        return expit(X.dot(self.w))

In [24]:
w_init = np.hstack([model.intercept_, model.coef_[0]])
em_classifier = EMClassifier(w_init)

In [25]:
em_classifier.fit(X_train, train, X_test, test)

 10%|█         | 1/10 [04:09<37:21, 249.09s/it]

Корреляция Спирмена: 0.7789951121519997
Корреляция Кендалла: 0.6072916743537586


 20%|██        | 2/10 [15:07<1:05:20, 490.06s/it]

Корреляция Спирмена: 0.780520698497128
Корреляция Кендалла: 0.6082327087948344


 30%|███       | 3/10 [19:28<44:58, 385.44s/it]  

...

Корреляция Спирмена: 0.7922366778898204
Корреляция Кендалла: 0.619939120501563


 90%|█████████ | 9/10 [56:25<05:50, 350.16s/it]

Корреляция Спирмена: 0.7922587228614142
Корреляция Кендалла: 0.6199611738108151


100%|██████████| 10/10 [1:00:25<00:00, 362.59s/it]

Корреляция Спирмена: 0.7939367884738594
Корреляция Кендалла: 0.6219811927245562


In [26]:
rating = pd.DataFrame({'player_id': unique_players,
                       'strength': em_classifier.w[1:1 + len(unique_players)]})
rating['name'] = rating['player_id'].map(players_names)
rating['questions_count'] = rating['player_id'].map(train.groupby('player_id')['question_id'].count())

rating.sort_values(by='strength', ascending=False).reset_index(drop=True).head(20)

,player_id,strength,name,questions_count
0,27403,7.882123,Максим Руссо,2131
1,4270,7.592833,Александра Брутер,2645
2,28751,7.459361,Иван Семушин,3682
3,27822,7.347681,Михаил Савченков,3087
4,30152,7.096111,Артём Сорожкин,4721
5,30270,6.997037,Сергей Спешков,3609
6,20691,6.656237,Станислав Мереминский,1548
7,7008,6.477289,Алексей Гилёв,4187
8,18036,6.466403,Михаил Левандовский,1365
9,87637,6.439265,Антон Саксонов,1122


5. А что там с вопросами? Постройте “рейтинг-лист” турниров по сложности вопросов.Соответствует ли он интуиции (например, на чемпионате мира в целом должны быть сложные вопросы, а на турнирах для школьников — простые)?

In [27]:
q_rating = dict(zip(unique_questions, em_classifier.w[-len(unique_questions):]))

train['difficulty'] = train['question_id'].map(q_rating)
train['tournament_name'] = train['tournament_id'].map(tournament_names)

tournaments_rating = train[['tournament_name', 'question_id', 'difficulty']].drop_duplicates()
tournaments_rating = tournaments_rating.groupby('tournament_name')['difficulty'].mean().sort_values().reset_index()

In [28]:
tournaments_rating.head(20)

,tournament_name,difficulty
0,Чемпионат Санкт-Петербурга. Первая лига,-5.164665
1,Угрюмый Ёрш,-3.900386
2,Первенство правого полушария,-3.491474
3,Синхрон высшей лиги Москвы,-3.311836
4,Воображаемый музей,-3.030560
5,Кубок городов,-3.024630
6,Ускользающая сова,-2.926947
7,Чемпионат России,-2.891351
8,Записки охотника,-2.849344
9,All Cats Are Beautiful,-2.682398


In [29]:
tournaments_rating.tail(20)

,tournament_name,difficulty
624,Кубок Оливье,2.476983
625,Осенняя кинолига,2.568301
626,Лига вузов. IV тур,2.568860
627,Синхрон-lite. Выпуск XXIX,2.590649
628,Синхрон простых вопросов. Зима,2.606232
629,"Ничто, нигде, никогда",2.627958
630,Маленькае люстэрка,2.643402
631,Лига Сибири. IV тур.,2.762877
632,(а)Синхрон-lite. Лига старта. Эпизод X,2.925593
633,(а)Синхрон-lite. Лига старта. Эпизод IV,2.947247
